# 概念 → 原始文档／图片 → 知识候选：逐算子调试

本版直接使用 **demiflow Dataset** 的 map、filter、join、reduce_by_key、group_batches 和文件 checkpoint；当前主线没有 SQLite。概念、文档、图片分别扩列，进入身份／联合提取时才嵌套资料。

默认 `MODE="view_saved"` 只读真实小批结果。改成 `execute`、另设一个新 RUN 后，可以逐 cell 执行；执行至哪一步就能查看哪一步。配置／源码变化需要新 run。显示的 limit 和 sample 不改变处理范围。

真实小批限定木兰、芦笙、Q1，5 个指定原始来源各最多 50,000 行。这是定向工程验证，不是随机校准。默认不调用模型、不出题。

In [ ]:
from pathlib import Path
import sys, asyncio, itertools, random, json
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from demiflow.standalone import local_data
from curation.v4.stream_flow import StreamFlow, ReadDocument, CleanDocument, CheckImage, model_input
from curation.v4.notebook_debug import show, detail
MODE = 'view_saved'
RUN = ROOT / 'state/curation/v4/demiflow_pilot_v2'
DATASET = ROOT / 'datasets/demiwtg'
SOURCES = [DATASET / p for p in ['meta/concepts.json', 'meta/qid_concepts.fat.jsonl.gz',
    'meta/docs.jsonl', 'meta/images.jsonl', 'corpus/pages-en-part1.jsonl.gz']]
flow = StreamFlow.open_saved(RUN) if MODE == 'view_saved' else StreamFlow(
    RUN, sources=SOURCES, ids=['legacy:木兰', 'legacy:芦笙', 'qid:Q1'], max_records_per_source=50000)
async def step(fn):
    if MODE == 'execute': return await asyncio.to_thread(fn)
def peek(name, limit=100, sample=False, seed=42, columns=None):
    rows = flow.saved(name).iter_rows()
    if sample: # reservoir: full scan, bounded display memory
        result=[]; rng=random.Random(seed)
        for i,row in enumerate(rows):
            if i<limit: result.append(row)
            else:
                j=rng.randrange(i+1)
                if j<limit: result[j]=row
    else: result=list(itertools.islice(rows,limit))
    show(result, limit=limit, columns=columns)
print('Mode:', MODE, '\nRun:', RUN)

## 1．读取原始文件，分别形成概念、文档、图片 Dataset

输入是 `datasets` 的采集文件，而非历史 clean_docs。每个文件按实际 schema 读取；同类文件才合并。Wiki 文档的 `lang + page_id` 与概念来源中的对应页面关联。源行读错、来源缺失、读取预算截断分别记录。

输出是三个 Dataset 的原始列；来源字段保留，新增处理字段保留明确含义：

| Dataset | 字段 | 含义 |
|---|---|---|
| concepts | concept_ref | `legacy:概念名` 或 `qid:QID`，当前来源概念引用，不是已消歧身份，也不迁移权威主键 |
| concepts | name / aliases / qid | 展示名、别名、外部 QID；完整原始身份字段另见 source_records |
| concepts | source_records | 原始身份行列表，每项 fields 是源字段，source 是定位 |
| concepts | identity_status | source_only 表示只沿用来源关联，未经身份核验 |
| concepts | page_refs | 原始 sitelink 展开的 lang、page_id、mapped_concept_ref；仅用于页面关联 |
| documents | doc_id | 来源版本及行内容生成的文档引用；不代表内容去重后的永久文档 ID |
| documents | concept_refs | 采集标签或页面对应得到的概念引用列表；允许共享资料 |
| documents | title / url / path | 原始标题、页面地址、已保存正文路径（原源不存在时可缺省） |
| documents | format / sections | saved_text 为已下载文本；wiki_sections 的正文在原始章节列表中 |
| documents | lang / page_id / source_qid | Wiki 语言、页面号（字符串）、来源自带 QID |
| documents | association_status | source_only 来源关联；unassociated 无关联；ambiguous_mapping 页面有多个待消歧对应 |
| images | image_id / concept_refs | 来源图片引用及采集时关联概念列表 |
| images | path / sha256 / url / caption | 原始字节位置、来源哈希、来源 URL、来源图注；均不能替代看图 |
| documents、images | source | 原始来源定位；其 path 为元数据文件，row 为从 1 开始的位置，snapshot 为文件版本，content_sha256 为源行对象哈希 |

文档、图片其余原始列原样保留，具体字段名随来源变化；概念原始列在 source_records.fields 完整保留。缺省字段不是空值事实。`page_refs`、关联键及异常文件属于计算辅助数据，无须作为业务主线逐表理解。

In [ ]:
await step(flow.prepare_sources)
peek('concepts', limit=3)
peek('documents', limit=3)
peek('images', limit=3)

## 2．筛选／采样概念，再关联文档和图片

输入：三个 Dataset 的原始列。概念先通过 ID 过滤、固定种子采样，输出增加 `selected`（是否入选）、`selection_reason`（selected、id_filter、concept_sample）。不设置 ids、sample_rate=1、取消读取上限，即使用同一条全量逻辑。

文档／图片分别将 concept_refs 展开为临时关联键，再 `.join(selected_concepts, how="semi")`。共享文档仍只保留一行供清洗。不会把 docs × images 联成大表。

输出 documents_selected、images_selected 保留原对象字段；documents_links / images_links 只含 concept_ref 和 doc_id / image_id。未匹配任何已读概念或关联歧义的资料保存在 *_unmatched，不丢弃；请求但没读到的概念在 missing_concepts。

In [ ]:
await step(flow.select)
show(flow.saved('concepts_selected').filter(lambda c:c['selected']).take(100))
peek('documents_selected', limit=3)
peek('missing_concepts')

## 3．读取下载正文 → 清洗文本外壳：文档连续扩列

输入：documents_selected；其中 path 或 sections 提供原文。

| 算子 | 新增输出字段 | 含义 |
|---|---|---|
| ReadDocument | raw_text / raw_sha256 | 完整读到的文本及读取字节的哈希；Wiki 是章节串接文本的哈希 |
| ReadDocument | read_status / read_error | readable 或 read_error；失败原因单独保存 |
| CleanDocument | clean_text / clean_version | 规则清洗文本及规则版本，不等于内容可靠 |
| CleanDocument | clean_status / clean_warnings | 清洗候选、待检查、不可用等状态及原因 |
| CleanDocument | clean_blocks | 块级原文与清洗文本的定位：block_id、raw_start/end、raw_text、clean_start/end、alignment；删除块保留原因 |
| CleanDocument | clean_counts | 输入／输出字符、块、删除等统计，展开查看具体计数 |

所有输入字段继续随行保留。清洗去掉能规则识别的外壳，不能仅因包含链接就删除知识，也不负责判定概念是否相关、来源是否可靠。`map_cached` 保存按输入与版本定位的结果；`checkpoint` 是执行和落盘边界。

In [ ]:
# 需要只调试读取时，可先单独执行并查看：
# read_ds = await asyncio.to_thread(lambda: flow.save('documents_read',
#     flow.saved('documents_selected').map_cached(ReadDocument(DATASET),
#         cache_dir=RUN/'cache/read_documents', version=flow.version)))
# peek('documents_read', limit=3)

# 正式文档链，后续清洗直接收到前一步扩列后的行：
documents = (flow.saved('documents_selected')
    .map_cached(ReadDocument(DATASET), cache_dir=RUN/'cache/read_documents', version=flow.version)
    .map_cached(CleanDocument(), cache_dir=RUN/'cache/clean_documents', version=flow.version))
await step(lambda: flow.save('documents_processed', documents))
peek('documents_processed', limit=3,
     columns=['doc_id','concept_refs','title','raw_text','clean_text','read_status','clean_status','clean_counts','clean_warnings'])

## 4．检查图片字节：图片独立扩列

输入：images_selected，重点使用 path、sha256。

输出保留全部输入字段，增加 byte_status（verified_bytes、not_local、hash_mismatch 等）和 byte_details（实际路径、解码／哈希检查结果、错误详情）。这一步核验文件可用性，不产生像素内容判断，也不证明图片支持哪条知识。

In [ ]:
images = flow.saved('images_selected').map_cached(
    CheckImage(DATASET), cache_dir=RUN/'cache/check_images', version=flow.version)
await step(lambda: flow.save('images_processed', images))
peek('images_processed', limit=5, columns=['image_id','concept_refs','path','caption','byte_status','byte_details'])

## 5．按概念统计资料覆盖：概念扩列

输入：入选概念、已完成的文档／图片 Dataset。通过 `.join()` 关联键并 `.reduce_by_key()` 计数，必须等两个分支都已落盘。

输出 concepts_ready 增加 document_count、image_count（关联资料数）、readable_documents、verified_images（读取成功／字节核验成功数）、material_status（有资料或当前读取范围内没资料）、knowledge_status（not_extracted）。这里的身份状态仍是 source_only。零资料不表示概念本身不存在，也不表示全网无资料。

In [ ]:
await step(flow.summarize)
peek('concepts_ready', columns=['concept_ref','identity_status','document_count','readable_documents','image_count','verified_images','material_status','knowledge_status'])

## 6．按概念分批汇集文档／图片：首次嵌套

输入：concepts_ready、documents_processed、images_processed。两条资料链独立处理完，再用 demiflow `.group_batches("concept_ref", max_rows=32)`，并左关联概念。

输出 knowledge_inputs 保留概念列，增加 materials、group_index、group_last。materials 每项含 concept_ref、doc_id 或 image_id、material_type（documents／images）、material（完整扩列文档／图片）。group_index 从 0 开始，group_last 表示该概念最后一批。无资料概念保留，可能没有 materials 字段。

这是执行分批，不是语义拆分；**跨批身份和知识联合整合尚未实现**，各批不能假装代表完整概念。大文档也仍需进一步实现章节提取与联合整合。

In [ ]:
await step(flow.gather)
show(flow.saved('knowledge_inputs').map(lambda r:{'concept_ref':r['concept_ref'], 'group_index':r.get('group_index'), 'group_last':r.get('group_last'), 'material_count':len(r.get('materials',[]))}).take(100))
# 按需查看指定概念的完整一批（包含完整长字段）：
# detail(flow.saved('knowledge_inputs').filter(lambda r:r['concept_ref']=='legacy:芦笙').take(1)[0])

## 7．查看身份算子的精确输入，再逐步运行知识算子

输入：一个概念及该批汇集资料。model_input 仅在这里转为旧知识算子的请求契约：case_id 为批任务哈希；concept_ref 保留业务关联；bundle 含试运行 concept_id、request（kind/value）及 materials；cleaned_materials 提供清洗版本；input_scope 明示批次和未做跨批整合。

concept_id 是来源引用生成的试运行 ID，不是跨来源身份合并结果。该兼容封装不会反过来作为三个上游 Dataset 的 schema。

In [ ]:
# 默认选 Q1 的单篇资料示例，避免把木兰的大批正文全部嵌入 notebook。
# 修改这里的 concept_ref 即可查看其他概念，保存的数据不受影响。
preview = flow.saved('knowledge_inputs').filter(lambda r:r['concept_ref']=='qid:Q1').map(model_input).take(1)
if preview: detail(preview[0])
RUN_MODELS = False
MODEL_CONFIG = {'max_calls':16} # 显式开启后才调用当前本地服务；配置冻结
async def knowledge_step(stage):
    if MODE == 'execute' and RUN_MODELS:
        await asyncio.to_thread(flow.knowledge, stage, MODEL_CONFIG)
    saved = RUN/'datasets'/f'knowledge_{stage}.jsonl'
    if saved.exists(): peek('knowledge_'+stage, limit=2)
    else: print(stage, '未执行；未产生模型请求。')

### 核对概念身份、判定资料归属

输入 cleaned_materials 和来源身份字段；输出 identity（status、target_label、reason、accepted_material_ids、rejected_materials、identity_groups、call）、identity_materials、identity_unexamined。blocked 记录阻塞原因。当前有限文字预览与图片元数据不能视作完整身份／像素审核。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

In [ ]:
await knowledge_step('identity')

### 去除重复资料、截取本次正文片段

输入身份接受的资料；输出 material_pack：passages、images、duplicates、omissions、image_gaps、coverage。passages 保留 source_id、text、start/end、original_chars、来源、清洗及原文定位；图片保留字节结果。当前预算选页和前缀截取仍需审查。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

In [ ]:
await knowledge_step('organize')

### 联合提取知识陈述

输入本次 passages；输出 extraction.facts、unresolved_conflicts、coverage_note 及调用记录。fact_id 标识候选，statement 是陈述，conditions/exceptions 是条件与例外，evidence 的 source_id/quote 是可定位引文。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

In [ ]:
await knowledge_step('extract')

### 比较重复、互补、条件差异与矛盾

输入同次提取多份来源及 extraction；输出 knowledge，保留 facts、unresolved_conflicts、changes、coverage_note。机器复核不能自动升级为核验知识；未解争议不应混进确定输出，现有模型仍可能违反这一点，需要审核。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

In [ ]:
await knowledge_step('consolidate')

### 核验图片对具体知识的支持

输入知识候选和可用图片；输出 image_evidence：图片内容描述、逐图片×知识的支持结果及调用信息。必须区分原图注、补充描述、支持区域／范围／局限；真实图像分支和 COS 仍未验收。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

In [ ]:
await knowledge_step('evidence')

### 保存知识候选与审核状态

输入前面累计结果；输出 export：case_id、concept_id、request、status、blocked、identity、facts、unresolved_conflicts、image_evidence、scope。同时保存 candidates JSON。状态仍是机器候选／阻塞，不是正式干净知识库验收，也不是 V4 试题。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

In [ ]:
await knowledge_step('export')

## 8．读取范围与断点

source_status 的 path/kind 是来源，rows 为实际扫描位置数，invalid_rows 为解析失败数，complete 表示是否读到文件末尾，status 区分 read、budget_limited、missing。完整错误原行在 parse_errors.jsonl。文件 checkpoint 与逐输入缓存都在本 run，失败 partial／error 文件保留。

可用 `peek('documents_processed', limit=100, sample=True, seed=42)` 抽样；抽样会遍历该已保存 Dataset，但只保留 limit 行。点击长字段展开全文，或 `detail(flow.saved('documents_processed').take(1)[0])` 查看单行。

In [ ]:
show([json.loads(p.read_text()) for p in sorted((RUN/'source_status').glob('*.json'))])
print('实际模型请求数:', len(list((RUN/'knowledge/calls').glob('*.request.json'))))

## 9．完整 pipeline 与最终输出抽样（最后一个执行 cell）

下面将前面已经调试的算子按同一顺序编排，使用同一个 StreamFlow 和 demiflow Dataset；没有另建框架。文档、图片仍独立扩列，然后按概念汇集。

`FINAL_THROUGH='gather'` 是当前实际运行终点，输出待核验资料；改为 `'export'` 可执行已接入的知识阶段，也可填某个中间阶段停下。模型调用仅在 `MODE='execute'` 且 through 指向知识阶段时发生，配置沿用 MODEL_CONFIG。

**本轮审核范围**：130篇处理后文档中，按概念×旧清洗状态分层，固定种子20260914每层最多2篇，共8篇；从17张图片按字节状态每层最多2张，共4张，其中2张本地图实际看图，2张缺失图未看像素。改动后检查相同ID，属于定向回归，不能当独立随机验收。新版本另抽不同种子20260915的6篇作补充检查。样本不是全库质量估计。

原文／清洗文本与全部定位保存在独立样本文件，下面展示概况、具体问题及抽样最终批次。没有产生知识候选时明确显示，不把 knowledge_inputs 称为干净知识库。

本轮清洗规则版本为 conservative-blocks/2：每个 clean_block 的 decision/reason 说明保留／排除及依据；clean_counts.excluded_nonblank 排除空白块后计数。identity_ineligible 是身份算子暂不采用的材料及理由；原始资料仍可回查。

In [ ]:
# 完整编排：沿用前面的算子，不新增调度框架。
def run_pipeline(f, through='gather', model_config=None):
    stages = ['identity', 'organize', 'extract', 'consolidate', 'evidence', 'export']
    if through not in ['gather', *stages]: raise ValueError('Unknown stopping stage')

    f.prepare_sources()  # 原始文件 -> concepts / documents / images；页面键 join
    f.select()           # 概念过滤/采样 -> 文档、图片分别 semi join

    documents = (f.saved('documents_selected')
        .map_cached(ReadDocument(f.dataset), cache_dir=f.run/'cache/read_documents', version=f.version)
        .map_cached(CleanDocument(), cache_dir=f.run/'cache/clean_documents', version=f.version))
    f.save('documents_processed', documents)  # JSONL checkpoint，失败可复用逐输入缓存

    images = (f.saved('images_selected')
        .map_cached(CheckImage(f.dataset), cache_dir=f.run/'cache/check_images', version=f.version))
    f.save('images_processed', images)

    f.summarize()  # join + reduce_by_key -> 概念资料计数
    f.gather()     # group_batches + left join -> 概念及本批资料
    if through == 'gather': return f.saved('knowledge_inputs')

    # 每一步内部都是 Dataset.map_async(对应知识算子).checkpoint(...)
    # 身份核对 -> 去重和截取片段 -> 联合提取 -> 冲突复核 -> 图片支持 -> 候选导出
    for stage in stages:
        result = f.knowledge(stage, config=model_config)
        if stage == through: return result

FINAL_THROUGH = 'gather'
if MODE == 'execute':
    final_dataset = await asyncio.to_thread(run_pipeline, flow, FINAL_THROUGH, MODEL_CONFIG)
else:
    final_dataset = flow.saved('knowledge_inputs' if FINAL_THROUGH == 'gather' else 'knowledge_'+FINAL_THROUGH)
    print('只读已保存结果；MODE=execute 时执行上面的完整编排。')

# 固定种子 reservoir 抽样：只在最终 Dataset 抽，不改变处理范围。
def sample_rows(dataset, limit=100, seed=20260914):
    result=[]; rng=random.Random(seed)
    for i,row in enumerate(dataset.iter_rows()):
        if i<limit: result.append(row)
        else:
            j=rng.randrange(i+1)
            if j<limit: result[j]=row
    return result
final_sample = sample_rows(final_dataset, limit=3)
if FINAL_THROUGH == 'gather':
    show([{'concept_ref':r['concept_ref'], 'group_index':r.get('group_index'),
           'group_last':r.get('group_last'), 'material_count':len(r.get('materials',[])),
           'documents':[{'doc_id':m['material']['doc_id'],'title':m['material'].get('title'),
                         'status':m['material']['clean_status']} for m in r.get('materials',[]) if m['material_type']=='documents'],
           'image_count':sum(m['material_type']=='images' for m in r.get('materials',[]))}
          for r in final_sample])
    print('当前最终输出：待核验资料批次；真实知识提取尚未执行。')
else:
    show(final_sample, limit=3)

review_path = RUN/'quality_review_v1/report.json'
if review_path.exists():
    review=json.loads(review_path.read_text())
    detail(review['scope'])
    detail(review['structural_checks'])
    show(review['document_reviews'], limit=100)
    show(review['image_reviews'], limit=100)
    show(review.get('holdout_reviews',[]), limit=100)
else:
    print('该 run 尚无质量审核报告，请检查本轮输出后单独保存审核结论。')
# 按需展开完整资料，不把整批长正文默认嵌入 notebook：
# detail(final_sample[0])
# peek('documents_processed', limit=100, sample=True, seed=20260915)
